# Lab 3. Surface code — 오류의 흔적을 읽고 논리 정보를 지키기

## 실습 소개

이번 Lab에서는 <strong>저장한 양자 정보를 직접 읽지 않고 오류의 흔적을 측정한 뒤, 그 결과에 따라 보정하는 과정</strong>을 살펴봅니다. 보정 뒤에는 검사 결과가 정상으로 돌아왔는지와 처음 정보가 복원되었는지를 따로 확인합니다.

복습자료의 3큐비트 반복 코드는 $\alpha|0\rangle+\beta|1\rangle$을 $\alpha|000\rangle+\beta|111\rangle$로 저장했습니다. 하나의 정보를 여러 물리 큐비트가 함께 이루는 상태에 담은 것이며, 같은 상태를 여러 개로 복제한 것은 아닙니다. 이렇게 저장한 양자 정보를 <strong>논리 큐비트</strong>라고 부릅니다. 두 논리 기저를 $|0_L\rangle,|1_L\rangle$로 쓰면 보호할 상태는

$$
|\psi_L\rangle=\alpha|0_L\rangle+\beta|1_L\rangle
$$

입니다. 아래첨자 L은 논리 상태를 뜻합니다.

데이터를 하나씩 측정하면 오류뿐 아니라 논리 성분까지 구별해 중첩을 잃을 수 있습니다. 대신 정상 논리 상태들이 공통으로 만족하는 관계를 측정합니다. 그 검사 연산자를 <strong>stabilizer</strong>, 여러 검사 결과를 순서대로 모은 기록을 <strong>신드롬(syndrome)</strong>이라고 부릅니다. 왜 이런 측정으로 정보를 남길 수 있는지는 문제 1의 작은 패리티 회로부터 확인하겠습니다.

이번에는 데이터 큐비트 9개에 논리 큐비트 하나를 저장하는 작은 <strong>surface code</strong>를 사용합니다. X형·Z형 검사 네 개씩으로 X·Y·Z 오류를 다룹니다. 데이터 번호와 검사 위치는 표로 제공하며, 논리 상태 준비와 전체 패치의 이상적 측정 계산은 제공 도구가 맡습니다.

신드롬에서 보정 연산을 고르는 고전 규칙을 <strong>decoder</strong>라고 부릅니다. 이는 인코딩 회로를 되돌리는 작업과 다릅니다. 데이터는 인코딩된 상태로 유지하고, decoder는 실제 오류나 목표 상태를 받지 않은 채 신드롬만으로 보정을 선택합니다. 처음 상태에 관한 정보는 보정이 끝난 뒤 검증할 때 사용합니다.

| 문제 | 할 일 |
|---|---|
| <strong>1. 정보를 남기는 측정</strong> | 보조 큐비트의 검사 결과와 측정 뒤 데이터 상태를 함께 확인합니다. |
| <strong>2. 오류와 신드롬</strong> | 검사와 오류의 관계로 어떤 검사 부호가 바뀌는지 계산합니다. |
| <strong>3. 신드롬에서 보정 선택하기</strong> | 허용된 오류 후보를 조사하고 서로 다른 보정이 동등한 이유를 설명합니다. |
| <strong>4. 논리 정보의 복원 검증</strong> | 정상 신드롬과 원래 논리 상태의 복원을 구분합니다. |

기본 실험에서는 정상 코드 상태에 <strong>최대 한 데이터 큐비트의 X·Y·Z 오류</strong>가 발생한다고 가정합니다. 상태 준비·검사·보정은 이상적입니다. 문제 4와 마지막 적용에서는 두 데이터 오류를 넣어, 이 가정을 벗어났을 때 결과를 어떻게 판단해야 하는지 살펴봅니다.

Lab1의 공동 상태 검증과 Lab2의 성능 비교처럼, 이번에도 확인한 사실과 그 사실로부터 내릴 수 있는 결론을 구분합니다. 목표는 오류 이름을 맞히는 것이 아니라 <strong>처음 저장한 논리 정보를 복원하는 것</strong>입니다.

## 노트북 사용 방법

각 절은 <strong>계산의 목적과 기호 정의, 핵심 수식, 풀이 예제, 직접 풀 문제</strong>로 이어집니다. 수식을 읽을 때에는 어떤 상태와 연산을 다루는지 먼저 확인하세요. 특히 정보를 저장하는 데이터 큐비트, 검사 결과를 읽는 보조 큐비트, 신드롬을 기록하는 고전 비트의 역할을 구분해서 읽습니다.

코드 셀은 위에서 아래로 차례대로 실행하세요. 앞에서 정의한 함수와 변수를 뒤에서 사용하므로, 중간부터 실행하면 오류가 날 수 있습니다. ‘제공 코드’로 표시된 셀도 실행해야 합니다. 여기에는 상태 준비, 오류 적용, 측정 결과 계산, 표 출력 도구가 들어 있습니다. 각 도구의 사용법은 필요한 곳에서 안내하므로, 내부 구현을 모두 분석한 뒤에 문제를 시작할 필요는 없습니다.

각 문제는 다음 순서로 풀어 보세요.

1. <strong>문제를 읽고 결과를 예상합니다.</strong> 어떤 정보를 측정하는지, 어떤 오류를 가정하는지, 보정에 사용할 수 있는 정보가 무엇인지 확인하세요. 예측을 요구하는 문항에서는 실행 전에 예상과 그 이유를 적습니다.
2. <strong>TODO 부분을 작성합니다.</strong> `TODO BEGIN`과 `TODO END` 사이의 `raise NotImplementedError`를 풀이로 교체하세요. 함수 이름과 매개변수는 유지하고, 문제에서 요구한 값을 반환합니다.
3. <strong>확인 셀과 연결 실험을 실행합니다.</strong> 답안 셀을 실행한 뒤 확인 셀을 실행하세요. 코드 검사는 실습 코드 실행 설정과 관계없이 작동합니다. 실행할 문항의 예상을 적은 뒤 `RUN_OBSERVATIONS = True`로 바꾸어 설정 셀을 다시 실행하고, 해당 문항의 실험 셀을 실행하세요. 코드를 수정했다면 답안·확인 셀과 그 함수를 사용하는 실험을 다시 실행해야 합니다.
4. <strong>결과를 해석합니다.</strong> 예상과 실행 결과를 비교하세요. 검사 비트가 맞는지, 측정 후 데이터가 어떤 상태인지, 처음 논리 정보가 복원되었는지를 구분해서 설명합니다.

확인 셀에는 다음과 같이 답안 상태가 표시됩니다.

- <strong>미완성:</strong> 아직 작성하지 않은 TODO가 있습니다.
- <strong>실패:</strong> 실행 오류가 발생했거나, 결과가 문제의 조건을 만족하지 않습니다. 메시지를 읽고 계산과 회로를 확인하세요.
- <strong>통과:</strong> 제공된 코드 검사를 통과했습니다.

일반 실습 코드는 처음에 `RUN_OBSERVATIONS = False`로 대기합니다. 함수를 모두 완성해도 자동으로 켜지지 않습니다. 설정을 True로 바꾼 뒤에는 새 문항의 예상을 적기 전에 전체 실행하지 말고 필요한 실험 셀만 실행하세요. 마지막 적용은 별도 `RUN_APPLICATION` 설정을 사용하므로, 일반 실습 코드의 실행을 허용했다고 함께 실행되지는 않습니다. 두 설정은 함수 TODO가 아닙니다.

서술형 답안은 자동 코드 검사와 별도로 작성합니다. 결과가 ‘통과’로 표시되더라도, 어떤 조건에서 무엇이 확인되었는지 설명할 수 있는지 점검하세요.

작은 측정 실험에서는 데이터 2개 또는 4개와 보조 큐비트 하나로 이루어진 Qiskit 회로를 사용합니다. 전체 패치 실험에서는 9개 데이터 상태에 대한 이상적인 stabilizer 측정을 제공 도구로 계산합니다. 작은 회로가 같은 측정 원리를 구현하는지 확인하지만, 보조 큐비트 여덟 개를 붙인 17큐비트 전체 하드웨어 회로를 작성하는 과제는 아닙니다.

이번 실습의 확률과 상태 일치도는 <strong>정확한 상태 계산</strong>으로 구합니다. 기본 오류 모형에서는 신드롬이 하나로 정해지므로, 작은 수치 오차를 제외하면 그 결과의 확률은 1입니다. 모든 상태에서 신드롬이 확정적인 것은 아니며, 작은 측정 예제에서는 여러 결과가 가능한 경우도 함께 살펴봅니다.

큐비트 번호와 비트 순서는 각 도구의 안내를 따르세요. 작은 회로의 로컬 번호와 전체 패치의 데이터 번호는 역할이 다릅니다. 신드롬 tuple도 검사 이름의 순서로 정리한 것이므로, 원시 측정 문자열과 같은 순서라고 가정하지 마세요.

이 Lab에서는 <strong>AI를 활용해 공부해도 좋습니다.</strong> 개념 설명을 요청하거나, 계산을 점검하거나, 오류 메시지의 원인을 찾는 데 활용하세요. AI가 제안한 코드도 직접 실행하고, 설명 문항에는 실제로 확인한 결과를 근거로 사용한 측정·보정·검증 방법이 왜 적절한지 자신의 말로 작성하세요.

## 실행 환경 준비

이 노트북은 Python 3.12, Qiskit 2.3.1, NumPy 2.2.6에서 검증했습니다. 로컬에서는 함께 배포된 `requirements.txt`를 설치한 환경을 Jupyter 커널로 선택하세요.

패키지가 없다면 새 코드 셀에서 다음 명령을 실행한 뒤 런타임을 다시 시작하고, 노트북을 위에서부터 실행하세요.

```python
%pip install qiskit==2.3.1 numpy==2.2.6
```

아래 제공 도구 셀도 실행합니다. 긴 제공 코드는 접혀 보일 수 있습니다. 각 문제에서 필요한 도구의 역할을 설명하므로, 내부 코드를 모두 분석한 뒤 시작할 필요는 없습니다. 작은 검사 회로는 실제 회로의 조건부 상태를 계산하고, 전체 패치의 검사는 9개 데이터 큐비트에 이상적인 projector를 적용합니다.

In [ ]:
import sys
from importlib.metadata import version
print("Python", sys.version.split()[0])
for package in ("qiskit", "numpy"):
    print(package, version(package))

### 제공 도구

회로 준비·실행·표 출력을 위한 도구입니다. 이 셀도 한 번 실행하세요. 공개 검사는 학습용 확인이며, 서술형 답안은 별도로 작성합니다.

In [ ]:
"""제공 도구: 작은 패리티 측정 회로와 9-data 코드의 이상적 측정·보정 실험.

전체 패치의 측정은 9개 데이터의 projector 계산입니다.
보조 큐비트 8개를 붙인 17큐비트 하드웨어 회로 실험은 아닙니다.
"""
import itertools
import numbers
from html import escape
import numpy as np
from IPython.display import HTML, display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

ATOL = 1e-9
N_DATA = 9
X_SUPPORTS = ((1, 2), (0, 1, 3, 4), (4, 5, 7, 8), (6, 7))
Z_SUPPORTS = ((0, 3), (1, 2, 4, 5), (3, 4, 6, 7), (5, 8))
CHECK_NAMES = ('a0','a1','a2','a3','b0','b1','b2','b3')
CHECK_STATUS = {}
CHECK_LABELS = ('1b · stabilizer 측정', '2b · 신드롬 예측', '3b · 보정 선택', '4a · 논리 fidelity')

def show_table(headers, rows):
    """문자열과 숫자를 HTML 표로 표시합니다. 이미지 출력은 만들지 않습니다."""
    def table_text(value):
        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"
        if isinstance(value, tuple):
            contents = ", ".join(table_text(item) for item in value)
            return "(" + contents + ("," if len(value) == 1 else "") + ")"
        if isinstance(value, list):
            return "[" + ", ".join(table_text(item) for item in value) + "]"
        return str(value)

    rows = list(rows)
    if any(len(row) != len(headers) for row in rows):
        raise ValueError("표의 열 수를 확인하세요.")
    heading = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(table_text(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

def _real(value, label):
    if (not isinstance(value, numbers.Real) or isinstance(value, (bool, np.bool_))
            or not np.isfinite(float(value))):
        raise AssertionError(f"{label}: 유한한 실수를 반환하세요.")
    return float(value)

def _real_tuple(value, length, label):
    if not isinstance(value, tuple) or len(value) != length:
        raise AssertionError(f"{label}: 길이 {length}의 tuple을 반환하세요.")
    return np.array([_real(item, label) for item in value], dtype=float)

def _close(actual, expected, message):
    if not np.allclose(actual, expected, atol=ATOL, rtol=0):
        raise AssertionError(message)

def _run_check(name, check):
    try:
        check()
    except NotImplementedError:
        CHECK_STATUS[name] = "미완성"
        print(f"{name}: 미완성 — TODO를 작성한 뒤 다시 실행하세요.")
        return False
    except Exception as exc:
        CHECK_STATUS[name] = "실패"
        print(f"{name}: 실패 — {type(exc).__name__}: {exc}")
        return False
    CHECK_STATUS[name] = "통과"
    print(f"{name}: 통과")
    return True

def _support_mask(support):
    return sum(1 << q for q in support)

def single_error_candidates():
    """무오류 한 개와 단일 data Pauli 27개. 특정 신드롬의 답은 고르지 않습니다."""
    return (("I", None),) + tuple((pauli, q) for q in range(N_DATA) for pauli in ("X", "Y", "Z"))

def _error_masks(error):
    assert isinstance(error, tuple) and len(error) == 2, "오류/보정은 (Pauli, qubit) tuple입니다."
    pauli, q = error
    if pauli == "I":
        assert q is None, "항등 연산은 ('I', None)입니다."
        return 0, 0
    assert pauli in ("X", "Y", "Z"), "Pauli는 I, X, Y, Z 중 하나입니다."
    assert isinstance(q, (int, np.integer)) and not isinstance(q, bool) and 0 <= q < N_DATA, "data 번호는 0부터 8까지입니다."
    bit = 1 << int(q)
    return (bit if pauli in ("X", "Y") else 0, bit if pauli in ("Z", "Y") else 0)

_CHECK_MASKS = tuple((_support_mask(s),0) for s in X_SUPPORTS) + tuple((0,_support_mask(s)) for s in Z_SUPPORTS)
_INDEX9 = np.arange(1 << N_DATA)
_PARITY9 = np.array([int(i).bit_count() % 2 for i in _INDEX9])

def _pauli_action(state, x_mask, z_mask):
    state = np.asarray(state, dtype=complex)
    assert state.shape == (512,), "9-data 상태벡터 길이는 512입니다."
    output = np.empty_like(state)
    phase = (1j ** (x_mask & z_mask).bit_count()) * (1 - 2 * _PARITY9[_INDEX9 & z_mask])
    output[_INDEX9 ^ x_mask] = phase * state
    return output

def apply_errors(state, errors):
    """errors의 Pauli들을 적힌 순서대로 적용합니다. 단일 오류도 list로 감쌉니다."""
    output = np.asarray(state, dtype=complex).copy()
    for error in errors:
        output = _pauli_action(output, *_error_masks(error))
    return output

def logical_state(label):
    """제공: 올바른 코드 상태. 인코더 회로 작성은 이번 과제가 아닙니다."""
    zero = np.zeros(512, dtype=complex)
    zero[0] = 1
    for support in X_SUPPORTS:
        zero = zero + _pauli_action(zero, _support_mask(support), 0)
    zero /= np.linalg.norm(zero)
    one = _pauli_action(zero, _support_mask((0, 3, 6)), 0)
    combinations = {"0": (1, 0), "1": (0, 1), "+": (1, 1), "-": (1, -1), "+i": (1, 1j), "-i": (1, -1j)}
    assert label in combinations, "논리 상태 이름은 0,1,+,-,+i,-i 중 하나입니다."
    alpha, beta = combinations[label]
    output = alpha * zero + beta * one
    return output / np.linalg.norm(output)

def ideal_syndrome_measurement(state):
    """제공: 이상적 projector로 모든 가능한 (syndrome, 확률, 사후상태)를 계산.

    각 결과 m에 (I+(-1)^m S)/2를 적용합니다. 기본 오류 모델에서는
    하나의 syndrome이 확률 1이지만, 일반 상태에서는 여러 결과도 지원합니다.
    """
    state = np.asarray(state, dtype=complex)
    assert state.shape == (512,) and np.isclose(np.vdot(state, state).real, 1.0, atol=ATOL, rtol=0), "정규화된 9-data 순수 상태가 필요합니다."
    branches = [((), state)]
    for x_mask, z_mask in _CHECK_MASKS:
        next_branches = []
        for prefix, vector in branches:
            transformed = _pauli_action(vector, x_mask, z_mask)
            for bit in (0, 1):
                projected = (vector + (-1)**bit * transformed) / 2
                if np.vdot(projected, projected).real > 1e-14:
                    next_branches.append((prefix + (bit,), projected))
        branches = next_branches
    results = []
    for syndrome, vector in branches:
        probability = float(np.vdot(vector, vector).real)
        results.append({"syndrome": syndrome, "probability": probability, "state": vector / np.sqrt(probability)})
    assert np.isclose(sum(row["probability"] for row in results), 1.0, atol=ATOL, rtol=0)
    return results

def deterministic_syndrome(state):
    """이번 실험처럼 신드롬이 확정적인 입력에서 사용합니다."""
    outcomes = ideal_syndrome_measurement(state)
    assert len(outcomes) == 1 and np.isclose(outcomes[0]["probability"], 1, atol=ATOL, rtol=0), "여러 신드롬이 가능한 입력입니다."
    return outcomes[0]["syndrome"]


def append_z_parity(qc, n_data):
    """제공된 Z 패리티 수집 부분: data 0..n_data-1 → ancilla n_data."""
    for q in range(n_data):
        qc.cx(q, n_data)


def ancilla_instrument(qc, n_data=None):
    """초기 ancilla 0과 마지막 Z 측정에 대한 데이터 연산자 K0,K1."""
    if not isinstance(qc, QuantumCircuit):
        raise AssertionError('QuantumCircuit을 반환하세요.')
    if n_data is None:
        n_data = qc.num_qubits - 1
    if n_data not in (2, 4) or qc.num_qubits != n_data + 1 or qc.num_clbits != 0:
        raise AssertionError('data 2개 또는 4개와 ancilla 하나, 고전 비트 없는 회로가 필요합니다.')
    if qc.parameters:
        raise AssertionError('미정 Parameter를 남기지 마세요.')
    forbidden = {'measure','reset','initialize','set_statevector','set_density_matrix','if_else','while_loop','for_loop','switch_case'}
    for instruction in qc.data:
        if instruction.operation.name in forbidden:
            raise AssertionError('상태 준비와 측정은 제공 도구가 담당합니다. 유니터리 부분만 반환하세요.')
    matrix = np.asarray(Operator(qc).data)
    if not np.isfinite(matrix).all() or not np.allclose(np.einsum('ji,jk->ik', matrix.conj(), matrix), np.eye(len(matrix)), atol=ATOL, rtol=0):
        raise AssertionError('유니터리 회로가 필요합니다.')
    size = 1 << n_data
    return matrix[:size, :size], matrix[size:, :size]


def _same_map(actual, expected):
    coefficient = np.vdot(expected, actual) / np.vdot(expected, expected)
    return bool(np.isclose(abs(coefficient), 1, atol=ATOL, rtol=0)
                and np.allclose(actual, coefficient * expected, atol=ATOL, rtol=0))


def measure_small_check(state, circuit_function, basis, n_data):
    state = np.asarray(state, dtype=complex)
    assert state.shape == (1 << n_data,) and np.isclose(np.vdot(state,state).real, 1)
    maps = ancilla_instrument(circuit_function(basis, n_data), n_data)
    branches = []
    for outcome, matrix in enumerate(maps):
        vector = matrix @ state
        probability = float(np.vdot(vector, vector).real)
        branches.append({'outcome': outcome, 'probability': probability,
                         'state': vector / np.sqrt(probability) if probability > 1e-14 else None,
                         'unnormalized_state': vector})
    assert np.isclose(sum(row['probability'] for row in branches), 1)
    return branches


def recovery_trial(logical_label, errors, decoder):
    """실제 오류는 실험 도구에만 주어집니다. decoder에는 신드롬만 전달합니다."""
    target = logical_state(logical_label)
    damaged = apply_errors(target, errors)
    outcomes = ideal_syndrome_measurement(damaged)
    assert len(outcomes) == 1, '이번 실험은 코드 상태에 확정적인 Pauli 오류를 적용합니다.'
    row = outcomes[0]
    correction = decoder(row['syndrome'])
    _error_masks(correction)
    corrected = apply_errors(row['state'], [correction])
    return {'target': target, 'damaged': damaged, 'postmeasurement': row['state'],
            'corrected': corrected, 'syndrome_before': row['syndrome'],
            'syndrome_after': deterministic_syndrome(corrected), 'correction': correction}


def check_check_circuit(function):
    def check():
        for basis in ('Z','X'):
            for n_data in (2,4):
                maps = ancilla_instrument(function(basis, n_data), n_data)
                observable = Operator.from_label(basis * n_data).data
                for m, actual in enumerate(maps):
                    target = (np.eye(1 << n_data) + (-1)**m * observable) / 2
                    assert _same_map(actual, target), '검사 비트와 사후 데이터 상태를 함께 확인하세요. 기저 변환의 앞뒤와 CX 방향도 확인하세요.'
    return _run_check(CHECK_LABELS[0], check)


def check_predict_syndrome(function):
    def check():
        fixtures = (
            ((), (0,0,0,0,0,0,0,0)),
            ((('X',4),), (0,0,0,0,0,1,1,0)),
            ((('Z',7),), (0,0,1,1,0,0,0,0)),
            ((('Y',2),), (1,0,0,0,0,1,0,0)),
            ((('X',1),('X',1)), (0,0,0,0,0,0,0,0)),
            ((('X',6),('X',7)), (0,0,0,0,0,0,0,0)),
        )
        for errors, expected in fixtures:
            actual = function(errors)
            assert isinstance(actual, tuple) and len(actual) == 8, '길이 8의 tuple을 반환하세요.'
            assert all(isinstance(v, numbers.Integral) and not isinstance(v,(bool,np.bool_)) and v in (0,1) for v in actual)
            assert actual == expected, '검사에 포함되는 위치, 오류 종류, 반교환 횟수의 홀짝을 확인하세요.'
    return _run_check(CHECK_LABELS[1], check)


def check_decode_syndrome(function):
    def check():
        code_basis = np.column_stack([logical_state('0'), logical_state('1')])
        for error in single_error_candidates():
            errors = [] if error[0] == 'I' else [error]
            observed = deterministic_syndrome(apply_errors(code_basis[:,0], errors))
            correction = function(observed)
            _error_masks(correction)
            assert correction in single_error_candidates(), '보정은 무오류 또는 단일 Pauli 후보 중 하나여야 합니다.'
            actual = np.column_stack([apply_errors(apply_errors(code_basis[:,j], errors), [correction]) for j in range(2)])
            assert _same_map(actual, code_basis), '선택한 보정이 두 논리 기저와 그 중첩을 보존하는지 확인하세요.'
    return _run_check(CHECK_LABELS[2], check)


def check_logical_fidelity(function):
    def check():
        plus = np.array([1,1],dtype=complex)/np.sqrt(2)
        plus_i = np.array([1,1j],dtype=complex)/np.sqrt(2)
        minus_i = np.array([1,-1j],dtype=complex)/np.sqrt(2)
        fixtures = (
            (plus_i, plus_i, 1.0), (plus_i, minus_i, 0.0),
            (plus, plus_i, 0.5), (plus_i, 1j*plus_i, 1.0),
            (np.array([1,0]), np.array([np.sqrt(3)/2,1j/2]), 0.75),
            (np.array([1,0,0,0]), np.array([0,0,0,1]), 0.0),
        )
        for target, output, expected in fixtures:
            actual = _real(function(target, output), 'fidelity')
            _close(actual, expected, '켤레복소수, 절댓값의 제곱, 전역위상 처리를 확인하세요.')
    return _run_check(CHECK_LABELS[3], check)

In [ ]:
LAB_RESULTS = {}


def run_demo(label, experiment, requires=(), requires_observations=True):
    LAB_RESULTS.pop(label, None)
    if requires_observations and not RUN_OBSERVATIONS:
        print(f"[대기] {label}: 예상을 적은 뒤 RUN_OBSERVATIONS를 True로 바꾸고 설정 셀과 이 실험을 실행하세요.")
        return None
    missing = [key for key in requires if CHECK_STATUS.get(key) != "통과"]
    if missing:
        print(f"[대기] {label}: 먼저 다음 코드 확인을 통과하세요: {', '.join(missing)}")
        return None
    try:
        result = experiment()
    except NotImplementedError:
        print(f"[대기] {label}: 앞의 TODO를 완성하세요.")
        return None
    except Exception as error:
        print(f"[실험 오류] {label}: {type(error).__name__}: {error}")
        return None
    LAB_RESULTS[label] = result
    return result


def small_state_text(state):
    if state is None:
        return "발생하지 않는 결과"
    return np.array2string(state, precision=4, suppress_small=True)


print("제공 도구 준비 완료. 신드롬 순서는 a0,a1,a2,a3,b0,b1,b2,b3입니다.")

In [ ]:
# 실행 전 예상을 적은 뒤 True로 바꾸고 이 셀을 다시 실행하세요.
RUN_OBSERVATIONS = False

## 문제 1. 데이터의 값을 읽지 않고 무엇을 측정할 수 있을까요?

이번 문제에서는 <strong>보조 큐비트로 패리티 한 비트를 읽고, 측정 뒤에도 데이터에 필요한 중첩을 남기는 회로</strong>를 만듭니다. 보조 큐비트에서 예상한 결과가 나오는지와, 데이터가 올바른 상태로 남는지를 함께 확인하겠습니다.

먼저 두 데이터의 값을 각각 측정하는 경우와 두 값의 관계만 측정하는 경우를 비교합니다. 그다음 이 측정을 stabilizer의 관점에서 설명하고, Z형 검사에서 X형 검사로 확장하겠습니다.

### 개별 값을 읽는 것과 관계를 읽는 것

두 데이터 큐비트의 상태가

$$
|\psi\rangle=\alpha|00\rangle+\beta|11\rangle,
\qquad |\alpha|^2+|\beta|^2=1
$$

이라고 합시다. 여기서 ket의 순서는 $|q_1q_0\rangle$입니다. 예를 들어 $|01\rangle$은 q1이 0, q0가 1인 성분을 뜻합니다.

두 데이터를 각각 Z 측정하면 00 또는 11 중 하나의 결과를 얻습니다. 각 결과의 확률은 $|\alpha|^2$, $|\beta|^2$입니다. 측정 뒤 데이터는 관측한 결과에 해당하는 상태로 남으므로, 처음 두 성분 사이에 있던 중첩은 유지되지 않습니다.

이번에는 개별 값 대신 <strong>두 값이 같은지 다른지</strong>만 확인하려 합니다. 두 비트의 XOR는 같으면 0, 다르면 1입니다. 이 한 비트를 <strong>패리티(parity)</strong>라고 부르겠습니다.

00과 11은 모두 패리티가 0입니다. 따라서 패리티만 읽으면 두 성분을 구분하지 않습니다. 이 점을 이용해, 두 성분 사이의 상대위상을 남기면서 오류의 흔적을 얻는 측정을 만들 수 있습니다.

이 관계를 관측량으로 나타내 보겠습니다. Z는 $|0\rangle$에 $+1$, $|1\rangle$에 $-1$의 부호를 줍니다. 두 위치에 작용하는 Z의 텐서곱을 $Z_0Z_1$이라고 쓰면, 두 부호의 곱은 다음과 같습니다.

| 데이터의 계산 기저 성분 | 두 Z 부호의 곱 | $Z_0Z_1$의 값 | 기록할 패리티 비트 |
|---|---|---|---:|
| $\lvert00\rangle$ | $(+1)(+1)$ | $+1$ | 0 |
| $\lvert11\rangle$ | $(-1)(-1)$ | $+1$ | 0 |
| $\lvert01\rangle$ | $(+1)(-1)$ | $-1$ | 1 |
| $\lvert10\rangle$ | $(-1)(+1)$ | $-1$ | 1 |

따라서 <strong>$Z_0Z_1$을 측정하는 것은 두 비트의 같음·다름에 해당하는 $+1/-1$ 결과를 얻는 것</strong>입니다. 이를 각각 비트 0과 1로 기록하겠습니다.

### 보조 큐비트에 패리티를 모으는 방법

데이터를 직접 측정하는 대신, 패리티만 보조 큐비트로 옮겨 보겠습니다.

보조 큐비트 $a$를 $|0\rangle$으로 준비합니다. 먼저 q0를 제어, $a$를 표적으로 하는 CX를 적용하고, 이어서 q1을 제어, 같은 $a$를 표적으로 하는 CX를 적용합니다.

데이터의 계산 기저 값이 $d_0,d_1$이면, 보조 큐비트는 각 값이 1일 때마다 뒤집힙니다. 처음 값이 0이므로 마지막에는

$$
0\oplus d_0\oplus d_1=d_0\oplus d_1
$$

이 기록됩니다. 두 데이터가 같으면 두 번 또는 한 번도 뒤집히지 않아 0이 되고, 다르면 한 번 뒤집혀 1이 됩니다.

이 과정이 중첩 상태에 어떻게 작용하는지 확인해 보겠습니다. 아래 상태식은 <strong>보조 큐비트를 먼저 적은 $|a\rangle\otimes|q_1q_0\rangle$ 순서</strong>입니다.

$$
\begin{aligned}
|0\rangle_a
\bigl(\alpha|00\rangle+\beta|11\rangle\bigr)
&\xrightarrow{\mathrm{CX}_{q_0\to a}}
\alpha|0\rangle_a|00\rangle
+\beta|1\rangle_a|11\rangle\\
&\xrightarrow{\mathrm{CX}_{q_1\to a}}
|0\rangle_a
\bigl(\alpha|00\rangle+\beta|11\rangle\bigr).
\end{aligned}
$$

첫 번째 CX 뒤에는 두 성분에 따라 보조 큐비트의 값이 다릅니다. 그러나 두 번째 CX까지 적용하면 두 성분 모두에서 보조 큐비트가 0이 됩니다. 보조 큐비트를 측정해도 00과 11 중 어느 성분인지는 구분되지 않으며, <strong>데이터의 계수와 상대위상은 그대로 남습니다.</strong>

이번에는 q0에 X 오류가 생겼다고 합시다. 데이터 상태는

$$
\alpha|01\rangle+\beta|10\rangle
$$

가 됩니다. 두 성분 모두 홀수 패리티이므로 같은 회로는

$$
|0\rangle_a
\bigl(\alpha|01\rangle+\beta|10\rangle\bigr)
\longmapsto
|1\rangle_a
\bigl(\alpha|01\rangle+\beta|10\rangle\bigr)
$$

를 만듭니다. 이번에는 보조 큐비트의 결과가 1이지만, 두 성분을 구별하지 않는다는 점은 같습니다.

이 검사로 패리티가 바뀌었다는 사실은 알 수 있습니다. 다만 q0와 q1 중 어느 위치의 X 오류인지까지 구분할 수는 없습니다. 오류에 관한 단서를 더 얻기 위해 여러 검사를 사용하는 방법은 다음 문제에서 다루겠습니다.

### 패리티 검사에서 stabilizer로

지금 살펴본 두 성분 $|00\rangle,|11\rangle$는 모두 $Z_0Z_1$의 값이 $+1$입니다. 따라서 그 중첩도

$$
Z_0Z_1
\bigl(\alpha|00\rangle+\beta|11\rangle\bigr)
=\alpha|00\rangle+\beta|11\rangle
$$

을 만족합니다.

오류 정정 코드에서는 이런 방식으로 <strong>보호하려는 논리 상태들이 공통으로 만족하는 검사 조건</strong>을 정합니다. 검사 연산자를 $S$, 인코딩된 논리 상태를 $|\psi_L\rangle$라고 쓰면 정상 조건은

$$
S|\psi_L\rangle=|\psi_L\rangle
$$

입니다. 정상 코드 상태를 그대로 두는 이런 Pauli 연산자를 stabilizer라고 부릅니다. 이를 관측량으로 측정하면 정상 상태에서 $+1$을 얻습니다.

여러 stabilizer가 있다면 정상 논리 상태는 그 조건들을 동시에 만족해야 합니다. 이 상태들이 이루는 공간이 <strong>코드 공간</strong>입니다.

정상 조건은 논리 상태를 하나로 정하는 조건이 아닙니다. 논리 0과 1, 그리고 그 중첩이 모두 같은 검사값을 가져야 합니다. 그래야 검사 결과를 얻는 과정에서 어느 논리 정보를 저장했는지 드러나지 않습니다.

### Z형 검사에서 X형 검사로 확장하기

데이터 네 개에서도 같은 방식으로 Z 패리티를 읽을 수 있습니다. 보조 큐비트를 0으로 준비하고, 각 데이터에서 같은 보조 큐비트로 CX를 적용하면 마지막에는

$$
d_0\oplus d_1\oplus d_2\oplus d_3
$$

이 기록됩니다. 이는 네 비트 중 1의 개수가 짝수인지 홀수인지를 나타냅니다. 제공 함수 `append_z_parity(qc, n_data)`가 이 패리티 수집 부분을 수행합니다.

X형 검사는 <strong>X 기저에서의 패리티</strong>를 읽습니다. X의 $+1$ 결과는 $|+\rangle$, $-1$ 결과는 $|-\rangle$에 대응합니다. 따라서 여러 데이터의 X 관측값을 곱하면, $|-\rangle$에 해당하는 성분의 개수가 짝수일 때 $+1$, 홀수일 때 $-1$이 됩니다.

이를 회로로 구현할 때에는 Lab1·Lab2에서 사용한 기저 변환을 재사용합니다.

$$
H|+\rangle=|0\rangle,\qquad
H|-\rangle=|1\rangle.
$$

각 데이터에 H를 적용하면 X 기저의 두 상태가 Z 기저의 두 상태로 옮겨집니다. 그 뒤에는 앞의 Z 패리티 수집 회로로 필요한 패리티를 보조 큐비트에 기록할 수 있습니다.

다만 이번에는 보조 큐비트만 측정하고, <strong>데이터는 뒤의 검사와 보정에 계속 사용</strong>합니다. 따라서 패리티를 보조 큐비트에 옮긴 다음에는 데이터의 기저를 원래대로 되돌려야 합니다. H는 두 번 적용하면 항등 연산이므로, 앞에서 H로 바꾼 데이터 기저는 뒤에서도 H를 적용해 되돌릴 수 있습니다.

| 단계 | Z형 검사 | X형 검사 |
|---|---|---|
| 측정 기준 준비 | 원래 기저 사용 | 각 데이터의 X 기저를 Z 기저로 변환 |
| 패리티 수집 | 제공된 Z 패리티 CX 회로 | 같은 Z 패리티 CX 회로 |
| 데이터의 기저 | 원래 기저 유지 | 원래 기저로 복원 |
| 결과 읽기 | 보조 큐비트 Z 측정 | 보조 큐비트 Z 측정 |

뒤쪽 기저 복원을 빠뜨리면 보조 큐비트의 결과 확률은 맞을 수 있습니다. 그러나 데이터가 변환된 기저에 남으므로, <strong>다음 단계로 전달되는 상태는 올바른 검사 뒤의 상태와 달라질 수 있습니다.</strong>

### 측정 결과뿐 아니라 사후 상태도 확인하기

이상적인 패리티 측정은 입력에서 측정 결과에 해당하는 성분만 남깁니다. 이를 수식으로 나타내는 연산자를

$$
P_0=\frac{I+S}{2},\qquad
P_1=\frac{I-S}{2}
$$

라고 하겠습니다. $I$는 데이터 전체에 대한 항등 연산입니다.

왜 이런 식이 되는지 살펴보겠습니다. $S$의 값이 $+1$인 상태를 $|\psi_+\rangle$, $-1$인 상태를 $|\psi_-\rangle$라고 하면,

$$
P_0|\psi_+\rangle
=\frac{|\psi_+\rangle+|\psi_+\rangle}{2}
=|\psi_+\rangle,
$$

$$
P_0|\psi_-\rangle
=\frac{|\psi_-\rangle-|\psi_-\rangle}{2}
=0
$$

입니다. 즉, $P_0$는 $+1$ 성분은 남기고 $-1$ 성분은 제거합니다. $P_1$은 반대로 $-1$ 성분을 남깁니다. 이런 연산자를 <strong>투영 연산자</strong>라고 부릅니다.

결과 비트를 $m$이라고 쓰면 두 경우를

$$
P_m=\frac{I+(-1)^mS}{2}
$$

로 묶을 수 있습니다. 입력 상태가 $|\psi\rangle$일 때, 결과 $m$의 확률은

$$
p_m=\|P_m|\psi\rangle\|^2
$$

이고, 그 확률이 0보다 크면 해당 결과에서의 데이터 상태는

$$
|\psi_m\rangle
=\frac{P_m|\psi\rangle}{\sqrt{p_m}}
$$

입니다. 결과 확률이 0이면 그 분기는 발생하지 않으므로 사후 상태를 정의하지 않습니다.

입력이 이미 한 패리티 안에 있으면, 그 패리티를 측정해도 상태가 유지됩니다. 반면 서로 다른 패리티 성분이 함께 있으면 결과에 따라 남는 성분이 달라집니다. 따라서 <strong>패리티 측정이 모든 입력 상태를 그대로 보존하는 것은 아닙니다.</strong> 이번에 구현할 회로는 각 결과의 확률과, 그 결과에 맞게 투영된 데이터 상태를 함께 재현해야 합니다.

### 직접 풀어보기

### (a) 개별 측정과 패리티 측정 비교하기

다음 두 데이터 상태를 사용합니다.

$$
|\psi\rangle
=\frac{\sqrt3}{2}|00\rangle
+\frac{i}{2}|11\rangle.
$$

ket의 순서는 $|q_1q_0\rangle$이고, 상태벡터의 기저 순서는 $|00\rangle,|01\rangle,|10\rangle,|11\rangle$입니다.

두 데이터를 각각 Z 측정한 경우에 가능한 결과, 각 결과의 확률, 측정 뒤의 데이터 상태를 적으세요. 이어서 측정 결과를 구분하지 않고 모은 상태를 각 분기 밀도행렬의 확률 평균으로 나타내세요. <strong>측정을 마친 뒤 기록을 무시하는 것만으로 처음의 상대위상이 복원되는지</strong> 설명합니다.

같은 입력에 보조 큐비트로 $Z_0Z_1$만 측정한 경우도 계산하세요. 어떤 결과가 나오고 데이터는 어떤 상태로 남는지 비교합니다.

| 측정 방법 | 얻는 결과와 확률 | 측정 후 데이터에 남는 정보 |
|---|---|---|
| 두 데이터를 각각 Z 측정 | … | … |
| 보조 큐비트로 ZZ 패리티만 측정 | … | … |

두 방법이 알아내는 정보의 차이가 중첩 보존과 어떻게 연결되는지 설명하세요.

### 1a 답안

계산 결과와 근거:

…

### (b) 검사 비트와 사후 데이터 상태를 함께 구현하기

`check_circuit(basis, n_data)`를 작성하세요.

- `basis`는 `"Z"` 또는 `"X"`입니다.
- `n_data`는 2 또는 4입니다.
- 반환값은 고전 비트와 측정이 없는 `QuantumCircuit(n_data + 1)`입니다.

회로 안의 큐비트 역할은 다음과 같습니다.

| `n_data` | 데이터의 로컬 번호 | 보조 큐비트의 로컬 번호 |
|---:|---|---:|
| 2 | q0, q1 | q2 |
| 4 | q0, q1, q2, q3 | q4 |

실행 도구가 데이터를 준비하고, 마지막 보조 큐비트를 $|0\rangle$으로 준비합니다. 여러분은 제공된 `append_z_parity(qc, n_data)`와 필요한 기저 변환을 연결해 <strong>측정 전의 유니터리 회로</strong>를 반환하세요. 상태 준비, 초기화, 측정 게이트는 함수 안에 넣지 않습니다.

실행 도구는 반환 회로 뒤에서 보조 큐비트를 Z 측정합니다. 결과 0은 지정한 패리티의 $+1$, 결과 1은 $-1$에 대응해야 합니다. 각 결과에서 데이터는 앞에서 정의한 $P_m$으로 얻는 사후 상태와 같아야 합니다.

확인 셀은 보조 큐비트의 결과 확률만 검사하지 않습니다. <strong>데이터에 불필요한 연산이 남지 않고 올바른 사후 상태가 만들어지는지도 확인</strong>합니다. 같은 측정 동작을 구현하는 동치 회로와, 각 결과의 상태 전체에 붙는 전역위상은 허용됩니다.

In [ ]:
def check_circuit(basis, n_data):
    qc = QuantumCircuit(n_data + 1)
    # TODO BEGIN — 지정한 패리티 검사와 사후 상태를 구현하세요.
    raise NotImplementedError("check_circuit")
    # TODO END
    return qc

In [ ]:
_ = check_check_circuit(check_circuit)

### 1b 답안

계산 결과와 근거:

…

### (c) 보조 큐비트의 결과만 맞으면 같은 검사일까요?

두 데이터에 대해 다음 두 회로를 비교합니다.

- 여러분이 작성한 정상적인 X형 검사
- 데이터를 원래 기저로 되돌리는 <strong>뒤쪽 H를 빠뜨린 회로</strong>

두 번째 회로는 결과를 비교할 수 있도록 제공합니다. 입력은

$$
|+\rangle_{q_1}|+\rangle_{q_0}
$$

입니다.

실행 전에 두 회로의 보조 큐비트 결과가 같은지 예상하세요. 실행 후에는 다음 표를 완성합니다.

| 회로 | 보조 큐비트 결과 0의 확률 | 결과 1의 확률 | 가능한 결과에서의 사후 데이터 상태 |
|---|---|---|---|
| 정상 X형 검사 | … | … | … |
| 뒤쪽 H를 빠뜨린 회로 | … | … | … |

두 회로가 같은 검사를 구현했다고 말할 수 있는지 설명하세요. 보조 큐비트의 결과와 데이터에 남은 상태를 구분해서 판단해야 합니다.

이어서 정상적인 ZZ 검사에

$$
|+\rangle_{q_1}|0\rangle_{q_0}
$$

를 입력합니다. 가능한 측정 결과의 확률과 각 결과에서의 사후 데이터 상태를 적으세요. 이 입력에서는 왜 측정 전후 상태가 항상 같다고 말할 수 없는지, 입력에 포함된 패리티 성분으로 설명합니다.

이번 두 데이터 실험은 패리티 측정의 원리를 확인하는 예제입니다. 이것만으로 오류 위치를 모두 구분하거나 임의의 단일 오류를 정정할 수 있다는 뜻은 아닙니다. 다음 문제에서는 같은 원리의 여러 검사를 surface-code 패치에서 함께 사용하겠습니다.

### 1c 답안

실행 전 예상:

…

계산·실행 결과와 근거:

…

### 실행 1c — 작성한 함수로 결과 확인

실행 전 예상을 적고 일반 실습 코드 실행 설정을 True로 바꾸어 다시 실행하세요. 앞의 코드 검사도 통과하면 이 실험이 실행됩니다. 결과를 확인한 뒤 답안에 실행 결과와 설명을 남기세요.

In [ ]:
def experiment_1c():
    def without_return_h(basis, n_data):
        qc = QuantumCircuit(n_data + 1)
        qc.h(range(n_data))
        append_z_parity(qc, n_data)
        return qc

    inputs = [
        ("정상 X 검사: ++ 입력", np.ones(4)/2, check_circuit, "X"),
        ("뒤쪽 H 누락: ++ 입력", np.ones(4)/2, without_return_h, "X"),
        ("정상 ZZ 검사: +0 입력", np.array([1,0,1,0])/np.sqrt(2), check_circuit, "Z"),
    ]
    results, rows = [], []
    for label, state, circuit_function, basis in inputs:
        branches = measure_small_check(state, circuit_function, basis, 2)
        results.append({"label": label, "branches": branches})
        for branch in branches:
            rows.append([label, branch["outcome"], branch["probability"], small_state_text(branch["state"])])
    show_table(["실험", "보조 큐비트 결과", "확률", "사후 데이터 벡터 (00,01,10,11)"], rows)
    return results

_ = run_demo("1c", experiment_1c, ("1b · stabilizer 측정",))

## 문제 2. 오류는 어떤 검사 결과를 바꿀까요?

앞 문제에서는 보조 큐비트로 패리티를 읽고, 측정 뒤 데이터가 어떤 상태로 남는지 확인했습니다. 이번에는 <strong>여러 검사를 함께 사용하고, 오류가 어떤 검사 결과를 바꾸는지 계산</strong>합니다.

먼저 사용할 surface-code 패치와 검사표를 살펴보겠습니다. 이어서 검사와 오류의 교환 관계로 신드롬을 예측하고, 그 예측을 실제 코드 상태의 이상적인 측정 결과와 비교하겠습니다.

### 여러 검사를 함께 사용해야 하는 이유

하나의 ZZ 검사만으로는 두 데이터 중 어느 위치에 X 오류가 생겼는지 구분하지 못했습니다. 어느 쪽에 X가 작용하더라도 두 비트의 패리티가 바뀔 수 있기 때문입니다.

오류에 관한 단서를 더 얻으려면 <strong>서로 다른 데이터 조합을 검사</strong>해야 합니다. 여러 검사에서 어떤 결과가 바뀌었는지 함께 보면, 한 검사만 사용할 때보다 오류 후보를 좁힐 수 있습니다.

위상 오류를 다루려면 검사 종류도 넓혀야 합니다. Z 오류는 Z 기저에서의 비트 값을 바꾸지 않으므로, ZZ 검사만으로는 그 흔적을 읽을 수 없습니다. 따라서 이번 코드에서는 Z형 검사와 함께 X형 검사도 사용합니다. 두 종류 검사가 각각 어떤 오류에 반응하는지는 뒤에서 연산자의 관계로 확인하겠습니다.

### 9개 데이터 큐비트의 검사표 읽기

이번 패치는 데이터 큐비트 9개를 다음과 같이 배치한 rotated surface code입니다.

|  |  |  |
|---|---|---|
| q0 | q1 | q2 |
| q3 | q4 | q5 |
| q6 | q7 | q8 |

이 표는 데이터의 위치와 번호를 나타냅니다. 실제로 어떤 데이터들을 함께 검사하는지는 다음 연산자 표로 정합니다.

| X형 검사 | 검사 연산자 | Z형 검사 | 검사 연산자 |
|---|---|---|---|
| $A_0$ | $X_1X_2$ | $B_0$ | $Z_0Z_3$ |
| $A_1$ | $X_0X_1X_3X_4$ | $B_1$ | $Z_1Z_2Z_4Z_5$ |
| $A_2$ | $X_4X_5X_7X_8$ | $B_2$ | $Z_3Z_4Z_6Z_7$ |
| $A_3$ | $X_6X_7$ | $B_3$ | $Z_5Z_8$ |

예를 들어 $A_0=X_1X_2$는 q1과 q2의 X 패리티를 읽는 검사입니다. $A_1=X_0X_1X_3X_4$는 q0, q1, q3, q4의 X 패리티를 읽습니다. 표에 적히지 않은 데이터에는 항등 연산이 작용합니다.

<strong>여러 데이터에 X가 적혀 있다고 해서, 그 데이터를 각각 X 측정해 개별 결과를 읽으라는 뜻은 아닙니다.</strong> 앞 문제처럼 보조 큐비트를 사용해 관측값들의 곱에 해당하는 패리티 한 비트를 읽고, 데이터는 다음 검사와 보정에 사용할 수 있도록 남깁니다.

한 검사가 작용하는 데이터 위치의 목록을 <strong>support</strong>라고 부르겠습니다. 위 표의 목록은 코드에서 `X_SUPPORTS`, `Z_SUPPORTS`로 제공됩니다. 각 목록의 순서는 검사 이름의 순서와 같습니다.

앞 문제의 작은 회로에서 사용한 번호는 로컬 번호였습니다. 예를 들어 데이터 네 개를 다루는 회로의 q0, q1, q2, q3는 그 회로 안의 자리입니다. 이번 패치의 $A_1$을 생각할 때에는 그 네 자리가 패치의 q0, q1, q3, q4에 대응합니다. <strong>작은 회로의 자리 번호와 패치의 고정된 데이터 번호를 구분</strong>하세요.

정상 코드 상태는 위의 여덟 검사에서 모두 $+1$을 갖습니다. 이 조건을 함께 만족하는 코드 공간에 논리 큐비트 하나를 저장합니다.

이 코드는 <strong>거리 3</strong>의 코드입니다. 검사에 드러나지 않으면서 논리 정보를 바꾸는 Pauli 연산이 적어도 데이터 세 개에 걸쳐 작용해야 한다는 뜻입니다. 이상적인 준비·측정·보정 아래에서는 임의의 단일 데이터 Pauli 오류를 정정할 수 있지만, 임의의 두 데이터 오류까지 정정한다고 보장할 수는 없습니다. 우선 단일 오류를 중심으로 검사 결과를 읽는 방법을 익히겠습니다.

### 검사와 오류의 순서를 바꾸면 무엇이 달라질까요?

검사 결과를 예측하려면, 오류가 검사 연산자와 어떤 관계를 갖는지 알아야 합니다.

두 연산자 $S,E$가

$$
SE=ES
$$

를 만족하면 <strong>교환한다</strong>고 합니다. 반대로

$$
SE=-ES
$$

를 만족하면 <strong>반교환한다</strong>고 합니다. 반교환하는 두 연산자는 순서를 바꾸었을 때 부호가 반대가 됩니다.

같은 큐비트에 작용하는 X와 Z는 반교환합니다.

$$
ZX=-XZ.
$$

반면 같은 종류의 Pauli끼리는 교환합니다. 서로 다른 큐비트에 작용하는 연산들도 교환합니다. 예를 들어 $i\ne j$이면

$$
X_iZ_j=Z_jX_i
$$

입니다.

이 관계가 측정 결과와 어떻게 연결되는지 살펴보겠습니다. 정상 코드 상태가

$$
S|\psi_L\rangle=|\psi_L\rangle
$$

를 만족하고, 그 상태에 오류 $E$가 작용했다고 합시다. 오류 뒤의 상태는 $E|\psi_L\rangle$입니다.

먼저 $S$와 $E$가 교환하면,

$$
SE|\psi_L\rangle
=ES|\psi_L\rangle
=E|\psi_L\rangle
$$

입니다. 따라서 오류 뒤의 상태도 이 검사에 대해서는 고윳값 $+1$을 갖습니다.

반대로 $S$와 $E$가 반교환하면,

$$
SE|\psi_L\rangle
=-ES|\psi_L\rangle
=-E|\psi_L\rangle
$$

입니다. 이번에는 오류 뒤의 상태가 검사 $S$의 고윳값 $-1$을 갖습니다.

여기서 나타난 부호는 <strong>검사를 측정했을 때 얻을 고윳값</strong>을 뜻합니다. 오류 상태에 전역위상을 붙여서 구분한다는 뜻은 아닙니다. 앞 문제의 투영 연산자로 보면, 이 상태는 $P_0=(I+S)/2$에서는 제거되고 $P_1=(I-S)/2$에서 그대로 남습니다.

따라서 <strong>어떤 검사와 반교환하는지 알면, 그 검사의 결과가 $+1$에서 $-1$로 바뀌는지 예측할 수 있습니다.</strong>

이 계산에는 논리 상태의 계수 $\alpha,\beta$가 필요하지 않았습니다. 정상 코드 상태에 정해진 Pauli 오류가 작용한 이번 조건에서는, 저장한 논리 정보를 알아내지 않고도 검사 부호의 변화를 계산할 수 있습니다.

### X형 검사와 Z형 검사를 함께 사용할 수 있는 이유

같은 위치의 X와 Z는 반교환하지만, 여러 위치에 작용하는 검사끼리는 겹치는 위치의 수를 함께 살펴봐야 합니다.

예를 들어

$$
S_X=X_iX_j,\qquad S_Z=Z_iZ_j
$$

를 비교하겠습니다. 두 검사에서 q$i$의 X와 Z를 교환할 때 부호가 한 번 바뀌고, q$j$에서도 한 번 더 바뀝니다. 따라서

$$
S_XS_Z=(-1)^2S_ZS_X=S_ZS_X
$$

입니다. 각 위치에서는 반교환하지만, 전체 연산자는 교환합니다.

이번 검사표의 X형 검사와 Z형 검사는 겹치는 위치가 없거나, 두 위치에서 겹칩니다. 따라서 서로 교환하도록 정해져 있습니다. 이상적인 측정에서는 앞에서 얻은 검사값을 다른 검사로 뒤집지 않으면서 여러 조건을 함께 확인할 수 있습니다.

여기서 두 관계를 구분하세요.

- <strong>검사끼리의 관계:</strong> 여러 정상 조건을 함께 측정할 수 있는지 확인합니다.
- <strong>검사와 오류의 관계:</strong> 오류가 어떤 검사 결과를 바꾸는지 확인합니다.

### 오류 종류에 따라 반응하는 검사가 달라집니다

X형 검사는 자신이 포함하는 위치의 X 오류와 교환하지만, Z 오류와는 반교환합니다. Z형 검사는 반대로 X 오류와 반교환하고 Z 오류와는 교환합니다.

Y는 전역위상을 제외하면 X와 Z를 함께 적용한 것과 같은 작용을 합니다. 따라서 X형 검사에서는 Z 성분이, Z형 검사에서는 X 성분이 검사 부호를 바꿉니다.

이를 정리하면 다음과 같습니다.

| 오류가 해당 검사에 포함된 데이터에 발생한 경우 | X형 검사 | Z형 검사 |
|---|---|---|
| X 오류 | 부호 유지 | 부호 반전 |
| Z 오류 | 부호 반전 | 부호 유지 |
| Y 오류 | 부호 반전 | 부호 반전 |

이 표는 <strong>오류가 발생한 위치가 그 검사에 포함될 때</strong> 적용합니다. 검사에 포함되지 않은 다른 데이터의 오류는 해당 검사와 교환하므로 그 부호를 바꾸지 않습니다.

오류가 여러 개이면 각 검사에서 생기는 부호 반전의 횟수를 셉니다. 홀수 번이면 부호가 바뀌고, 짝수 번이면 원래 부호로 돌아옵니다. 예를 들어 하나의 Z형 검사에 포함된 서로 다른 두 위치에 X 오류가 생기면, 그 검사에서는 부호 반전 두 번이 서로 상쇄될 수 있습니다.

따라서 오류가 있는 위치와 겹친다는 이유만으로 검사 비트를 모두 1로 적어서는 안 됩니다. <strong>검사에 포함되는 위치, 오류의 종류, 반교환 횟수의 홀짝</strong>을 함께 확인해야 합니다.

### 검사 부호를 신드롬 비트로 기록하기

각 검사에서 $+1$을 얻으면 0, $-1$을 얻으면 1로 기록합니다. X형 검사의 네 비트를 $a_0,a_1,a_2,a_3$, Z형 검사의 네 비트를 $b_0,b_1,b_2,b_3$라고 쓰겠습니다.

신드롬의 순서는 항상

$$
s=(a_0,a_1,a_2,a_3,b_0,b_1,b_2,b_3)
$$

입니다.

이 tuple은 <strong>검사 이름의 순서로 정리한 기록</strong>입니다. Qiskit의 원시 비트 문자열을 그대로 복사한 순서가 아니며, 제공 도구가 이 순서로 결과를 정리해 줍니다. 여기서 $a_i,b_i$는 앞 Lab의 게임 응답이 아니라 각각의 stabilizer 측정 기록입니다.

### 함께 예측하기: 중앙 데이터의 X 오류

가운데 데이터 q4에 $X_4$ 오류가 생겼다고 합시다.

X형 검사들은 $X_4$와 교환합니다. q4를 포함한 X형 검사에서도 같은 종류의 Pauli가 만나므로 부호가 바뀌지 않습니다. 따라서 $a_0,a_1,a_2,a_3$는 모두 0입니다.

Z형 검사에서는 q4를 포함하는 검사를 찾습니다.

$$
B_1=Z_1Z_2Z_4Z_5,\qquad
B_2=Z_3Z_4Z_6Z_7
$$

가 q4를 포함합니다. 이 두 검사에는 q4의 Z가 있고, 오류는 같은 위치의 X이므로 부호가 뒤집힙니다. q4를 포함하지 않는 $B_0,B_3$의 부호는 그대로입니다.

따라서

$$
s(X_4)=(0,0,0,0,0,1,1,0)
$$

입니다.

이 기록은 q4의 값을 직접 측정한 결과가 아닙니다. <strong>어떤 검사 조건이 달라졌는지 모아 놓은 결과</strong>입니다. 같은 방식으로 다른 위치와 다른 종류의 오류도 분석할 수 있습니다.

### 예측과 실제 측정 계산 비교하기

신드롬 예측 함수는 검사표와 오류의 관계를 사용합니다. 이와 별도로, 제공 측정 도구는 오류가 적용된 실제 9-data 상태에

$$
P_m=\frac{I+(-1)^mS}{2}
$$

를 적용해 결과의 확률과 사후 상태를 계산합니다. 여덟 검사를 차례로 수행하여 신드롬을 얻습니다.

앞 문제에서 만든 작은 측정 회로가 이 투영 연산자를 구현하는지는 이미 확인했습니다. 이번에는 같은 원리로 전체 패치의 검사 결과를 계산합니다. <strong>학생의 예측값을 그대로 측정 결과로 사용하는 것이 아니므로</strong>, 두 결과를 비교해 신드롬 계산을 점검할 수 있습니다.

논리 상태의 준비와 큰 상태벡터의 처리는 제공 도구가 맡습니다. 여러분은 검사표를 읽고, 오류와 검사 사이의 관계를 신드롬으로 옮기는 일에 집중하세요.

### 직접 풀어보기

### (a) 검사표에서 겹치는 위치와 오류의 흔적 읽기

먼저 $A_1$과 $B_1$이 함께 작용하는 데이터 위치를 찾으세요. 그 위치들에서 생기는 부호 반전을 세어, 두 검사 전체가 교환하는 이유를 설명합니다.

이어서 q4의 오류를 각각 Z와 Y로 바꾸었을 때의 신드롬을 예측하세요. 먼저 부호가 바뀌는 검사 이름을 적고, 정해진 순서의 8비트 tuple로 옮깁니다.

| 오류 | 부호가 바뀌는 검사 | 예측한 신드롬 |
|---|---|---|
| $Z_4$ | … | … |
| $Y_4$ | … | … |

각 행에서 오류 종류와 검사 종류의 관계를 설명하세요. 오류의 이름과 같은 종류의 검사가 반응한다고 바로 대응시키지 말고, 교환·반교환 관계를 사용해야 합니다.

### 2a 답안

계산 결과와 근거:

…

### (b) 신드롬을 예측하는 함수 작성하기

`predict_syndrome(errors)`를 작성하세요.

`errors`는 `(Pauli, 데이터 번호)`를 원소로 갖는 목록 또는 tuple입니다. Pauli는 `"X"`, `"Y"`, `"Z"` 중 하나이고, 데이터 번호는 0부터 8까지입니다.

예를 들어

```python
[("X", 4)]
```

는 q4의 X 오류를 뜻합니다. 빈 목록이나 빈 tuple은 오류가 없다는 뜻입니다. 오류가 여러 개 주어질 수도 있고, 같은 위치가 여러 번 나타날 수도 있습니다.

함수는 각 검사와의 반교환 횟수를 반영하여 <strong>정수 0과 1로 이루어진 길이 8의 tuple</strong>을 반환해야 합니다. 반환 순서는 항상 다음과 같습니다.

```python
(a0, a1, a2, a3, b0, b1, b2, b3)
```

제공된 `X_SUPPORTS`, `Z_SUPPORTS`를 사용하세요. 중앙 X 오류의 결과만 반환하지 말고, 오류 위치와 종류가 달라져도 계산하도록 작성합니다. 여러 오류가 같은 검사에 영향을 줄 때에는 부호 반전의 홀짝도 반영해야 합니다.

이 함수는 <strong>알려진 오류를 분석하거나 보정 후보를 조사하는 함수</strong>입니다. 다음 문제에서 작성할 decoder에 실제 오류가 알려져 있다는 뜻은 아닙니다. 또한 여러 오류의 신드롬을 예측할 수 있다는 것과, 그 오류들을 모두 정정할 수 있다는 것은 다른 주장입니다.

In [ ]:
def predict_syndrome(errors):
    # TODO BEGIN — 검사표와 오류 목록으로 8비트 tuple을 반환하세요.
    raise NotImplementedError("predict_syndrome")
    # TODO END

In [ ]:
_ = check_predict_syndrome(predict_syndrome)

### 2b 답안

계산 결과와 근거:

…

### (c) 예측과 실제 검사 결과 비교하기

제공 도구로 다음 오류들을 코드 상태에 각각 적용하고, 이상적인 stabilizer 측정 결과를 확인하세요. 논리 입력은 상대위상을 포함하는

$$
|+i_L\rangle
=\frac{|0_L\rangle+i|1_L\rangle}{\sqrt2}
$$

를 사용합니다.

먼저 각 오류의 신드롬을 예상하고, 작성한 함수의 출력과 측정 도구의 결과를 비교하세요.

| 실제 오류 | 함수가 예측한 신드롬 | 제공 측정 도구의 신드롬 |
|---|---|---|
| $Z_4$ | … | … |
| $Y_4$ | … | … |
| $X_1$ | … | … |
| $X_2$ | … | … |

두 계산이 일치하는지 확인하세요. 일치하지 않는 행이 있다면 해당 위치를 포함하는 검사, Pauli 종류에 따른 반교환, 비트의 반환 순서를 차례로 점검합니다.

이어서 서로 다른 오류가 같은 신드롬을 만드는 행이 있는지 찾아보세요. 그 결과를 근거로 <strong>신드롬이 언제나 실제 오류의 위치와 종류를 하나로 정해 준다고 말할 수 있는지</strong> 설명합니다.

다음 문제에서는 같은 신드롬에 여러 후보가 대응하더라도, 원래의 논리 정보를 복원하는 보정을 선택할 수 있는지 살펴보겠습니다.

### 2c 답안

실행 전 예상:

…

계산·실행 결과와 근거:

…

### 실행 2c — 작성한 함수로 결과 확인

실행 전 예상을 적고 일반 실습 코드 실행 설정을 True로 바꾸어 다시 실행하세요. 앞의 코드 검사도 통과하면 이 실험이 실행됩니다. 결과를 확인한 뒤 답안에 실행 결과와 설명을 남기세요.

In [ ]:
def experiment_2c():
    target = logical_state("+i")
    rows, results = [], []
    for error in (("Z",4), ("Y",4), ("X",1), ("X",2)):
        prediction = predict_syndrome([error])
        observed = deterministic_syndrome(apply_errors(target, [error]))
        rows.append([error, prediction, observed, prediction == observed])
        results.append({"error": error, "prediction": prediction, "observed": observed})
    show_table(["실제 오류", "학생의 예측", "독립 상태 측정", "일치"], rows)
    return results

_ = run_demo("2c", experiment_2c, ("2b · 신드롬 예측",))

## 문제 3. 같은 신드롬에 여러 오류가 대응하면 무엇을 보정해야 할까요?

앞 문제에서는 서로 다른 오류가 같은 신드롬을 만들 수 있다는 것을 확인했습니다. 이번에는 <strong>신드롬만 받아 보정 연산을 선택하는 decoder</strong>를 작성합니다.

실제 오류를 하나로 알아내지 못하면 보정도 불가능한지 먼저 살펴보겠습니다. 그다음 보정의 성공 기준을 이용해 후보를 선택하고, 무오류와 모든 단일 데이터 Pauli 오류에서 논리 정보가 복원되는지 확인하겠습니다.

### 보정의 성공은 무엇으로 판단해야 할까요?

정상 논리 상태를 $|\psi_L\rangle$, 실제로 발생한 오류를 $E$라고 쓰겠습니다. 오류가 발생한 뒤의 데이터 상태는

$$
E|\psi_L\rangle
$$

입니다.

앞 문제에서 다룬 것처럼, 정상 코드 상태에 정해진 Pauli 오류가 작용하면 각 stabilizer의 검사값이 정해집니다. 이상적인 신드롬 측정은 그 검사값을 읽으며, 이미 해당 검사값을 갖는 데이터 상태를 그대로 남깁니다.

이후 보정 연산 $R$을 적용하면 데이터는

$$
RE|\psi_L\rangle
$$

가 됩니다. 오른쪽의 $E$가 먼저 작용하고, 왼쪽의 $R$이 나중에 작용합니다. 따라서 보정의 효과를 판단하려면 <strong>오류와 보정을 합친 연산 $RE$</strong>가 논리 상태에 무엇을 하는지 확인해야 합니다.

가장 간단한 경우는 발생한 오류와 같은 Pauli를 같은 위치에 다시 적용하는 것입니다. 예를 들어 실제 오류가 $X_i$이고 보정도 $X_i$이면,

$$
X_iX_i=I
$$

이므로 원래 상태로 돌아옵니다. Y와 Z도 각각 두 번 적용하면 항등 연산이 됩니다.

하지만 이것만이 가능한 보정은 아닙니다. <strong>보정 뒤 남는 연산이 정상 논리 상태를 바꾸지 않는다면, 실제 오류와 다른 연산을 선택해도 복원에 성공합니다.</strong>

### Stabilizer가 남아도 논리 상태는 같습니다

정상 코드 상태는 각 stabilizer $S$에 대해

$$
S|\psi_L\rangle=|\psi_L\rangle
$$

를 만족합니다. 따라서 오류와 보정을 합친 연산이 stabilizer라면,

$$
RE|\psi_L\rangle
=S|\psi_L\rangle
=|\psi_L\rangle
$$

입니다.

여러 stabilizer를 곱한 연산도 같은 성질을 갖습니다. 예를 들어 $S_1,S_2$가 각각 정상 논리 상태를 그대로 둔다면,

$$
S_1S_2|\psi_L\rangle
=S_1|\psi_L\rangle
=|\psi_L\rangle
$$

입니다. 따라서 $RE$가 stabilizer 하나이거나 그 곱이면 논리 정보가 복원됩니다. 상태 전체에 같은 전역위상만 붙은 경우도 같은 물리적 상태로 취급합니다.

여기서 중요한 것은 특정 논리 입력 하나만 우연히 같아지는 것이 아니라, <strong>코드 공간 안의 모든 논리 상태에 대해 같은 성질이 성립한다는 점</strong>입니다.

### 함께 살펴보기: X1 오류에 X2로 보정하기

앞 문제에서 $X_1$과 $X_2$는 모두 $B_1$의 부호만 바꾸었습니다. 두 오류의 신드롬은 같습니다.

$$
s(X_1)=s(X_2)
=(0,0,0,0,0,1,0,0).
$$

따라서 이 신드롬만 받은 decoder는 실제 오류가 $X_1$이었는지 $X_2$였는지 구분할 수 없습니다.

이때 보정으로 $X_2$를 선택했다고 해 봅시다. 실제 오류가 $X_2$였다면

$$
X_2X_2=I
$$

이므로 원래 상태로 돌아옵니다.

실제 오류가 $X_1$이었다면 남는 연산은

$$
X_2X_1=X_1X_2=A_0
$$

입니다. 서로 다른 위치의 X는 교환하며, $A_0=X_1X_2$는 이번 코드의 stabilizer입니다. 따라서 이 경우에도

$$
X_2X_1|\psi_L\rangle
=A_0|\psi_L\rangle
=|\psi_L\rangle
$$

가 됩니다.

| 실제 오류 $E$ | 선택한 보정 $R$ | 남는 연산 $RE$ | 정상 논리 상태에 대한 작용 |
|---|---|---|---|
| $X_1$ | $X_2$ | $A_0$ | 상태를 그대로 둡니다. |
| $X_2$ | $X_2$ | $I$ | 상태를 그대로 둡니다. |

같은 보정 $X_2$가 두 경우 모두에서 성공합니다. 반대로 보정으로 $X_1$을 선택해도 두 경우 모두 논리 상태를 복원할 수 있습니다.

이처럼 <strong>같은 논리 정보를 복원하는 보정들을 동등한 보정으로 취급</strong>합니다. 오류의 물리적인 위치를 유일하게 알아내는 일과, 저장한 논리 정보를 복원하는 일이 항상 같은 문제는 아닙니다.

### 같은 신드롬의 후보를 골라도 되는 범위

그렇다면 같은 신드롬을 만드는 연산이면 무엇이든 보정으로 골라도 될까요? 여기서는 <strong>어떤 오류 후보를 허용했는지</strong>가 중요합니다.

현재 기본 오류 모형은 무오류 또는 한 데이터 큐비트의 X·Y·Z 오류입니다. 실제 오류와 보정 후보가 모두 이 범위에 있으면, 두 연산을 합친 결과는 많아야 데이터 두 개에 걸쳐 작용합니다.

두 연산의 신드롬이 같다면, 각 검사에서 오류가 바꾼 부호를 보정이 다시 바꾸므로 모든 검사 부호가 정상으로 돌아옵니다. 여기에 이번 코드의 거리 3이라는 성질을 함께 사용합니다. 앞 문제에서 설명했듯이, <strong>검사에 드러나지 않으면서 논리 정보를 바꾸는 Pauli 연산은 적어도 데이터 세 개에 걸쳐 작용해야 합니다.</strong>

따라서 이 범위의 두 후보를 합친 연산은 검사 결과를 정상으로 유지하면서 논리 정보를 바꾸는 연산이 될 수 없습니다. 이번 코드에서는 같은 신드롬을 가진 단일 오류 후보들이 stabilizer만큼 차이 나며, 어느 후보로 보정해도 논리 정보를 복원합니다.

이 설명은 <strong>실제 오류도 최대 한 데이터 Pauli 오류라는 가정 안에 있을 때의 보장</strong>입니다. 그 가정을 벗어나면 같은 신드롬의 단일 후보를 골랐다는 사실만으로 복원 성공을 결론 낼 수 없습니다. 그 경우는 다음 문제에서 확인하겠습니다.

### 신드롬으로 보정 후보를 찾는 과정

무오류는 한 가지이고, 데이터 9개 각각에서 가능한 단일 Pauli 오류는 X·Y·Z 세 가지입니다. 따라서 조사할 후보는

$$
1+9\times3=28
$$

개입니다.

제공 함수 `single_error_candidates()`가 이 후보들을 반환합니다. 각 후보는 다음과 같이 표현합니다.

| 후보의 표현 | 의미 |
|---|---|
| `("I", None)` | 아무 보정도 하지 않는 항등 연산 |
| `("X", 4)` | q4에 X를 적용 |
| `("Y", 2)` | q2에 Y를 적용 |
| `("Z", 5)` | q5에 Z를 적용 |

이 후보들을 조사하는 과정은 다음과 같습니다.

1. 후보 하나를 고릅니다.
2. 앞에서 작성한 `predict_syndrome()`로 그 후보가 만들 신드롬을 계산합니다.
3. 실제로 받은 신드롬과 비교합니다.
4. 두 신드롬이 같으면 그 후보를 보정으로 선택합니다.

같은 신드롬의 후보가 여러 개이면 그중 하나를 선택하면 됩니다. 후보가 28개라는 것이 서로 다른 신드롬도 반드시 28개라는 뜻은 아닙니다.

여기서 <strong>실제 오류를 알고 예측하는 것과, 가능한 후보를 조사하는 것을 구분</strong>하세요. decoder는 실제 오류를 전달받지 않습니다. 검사표와 후보 목록을 알고 있으므로, 각 후보가 실제 오류였다면 어떤 신드롬이 나왔을지 계산하는 것입니다.

| 역할 | 사용할 수 있는 정보 |
|---|---|
| 오류를 주입하는 실험 도구 | 처음 준비한 상태와 주입한 실제 오류 |
| 후보의 신드롬을 계산하는 함수 | 조사 중인 후보와 검사표 |
| decoder | 측정으로 얻은 신드롬, 코드의 검사표와 후보 목록 |
| 복원 결과를 확인하는 검증 도구 | 처음 상태, 선택한 보정, 보정 후 상태 |

이번 decoder는 보정 연산의 이름을 반환하고, 제공된 실행 도구가 그 연산을 데이터에 적용합니다. 데이터를 인코딩된 상태로 유지하며 보정하므로, 인코딩 회로를 되돌리는 작업과는 구분해야 합니다.

### 논리 기저뿐 아니라 중첩도 보존해야 합니다

코드가 논리 큐비트 하나를 저장한다면 임의의 논리 상태는

$$
|\psi_L\rangle
=\alpha|0_L\rangle+\beta|1_L\rangle
$$

로 나타낼 수 있습니다.

어떤 보정이 두 논리 기저에 대해

$$
RE|0_L\rangle=e^{i\phi}|0_L\rangle,
\qquad
RE|1_L\rangle=e^{i\phi}|1_L\rangle
$$

처럼 <strong>같은 전역위상</strong>을 남긴다면, 선형성에 의해

$$
\begin{aligned}
RE|\psi_L\rangle
&=\alpha RE|0_L\rangle+\beta RE|1_L\rangle\\
&=e^{i\phi}
\bigl(\alpha|0_L\rangle+\beta|1_L\rangle\bigr)
\end{aligned}
$$

입니다. 따라서 그 중첩도 같은 물리적 상태로 복원됩니다.

반면 두 기저에 서로 다른 위상이 붙으면, 각각의 기저 상태만 검사했을 때에는 변화가 드러나지 않을 수 있지만 중첩의 상대위상은 달라질 수 있습니다. 그래서 제공 코드 확인은 두 논리 기저가 각각 같은 상태처럼 보이는지만 확인하지 않고, <strong>두 기저에 대한 작용이 하나의 공통 전역위상으로 설명되는지</strong>도 확인합니다.

신드롬이 정상으로 돌아왔다는 사실과, 이런 의미에서 논리 정보가 복원되었다는 사실은 구분해야 합니다.

### 직접 풀어보기

### (a) 다른 보정이 같은 논리 상태를 복원하는지 판단하기

앞에서 작성한 함수나 검사표를 사용해 $Z_0$와 $Z_3$의 신드롬을 각각 구하세요.

실제 오류가 $Z_0$인데 보정으로 $Z_3$를 선택했다고 하겠습니다. 다음 내용을 적으세요.

- 두 오류가 같은 신드롬을 만드는가?
- 실제 오류 뒤에 선택한 보정을 적용하면 어떤 연산이 남는가?
- 그 연산은 검사표의 어떤 stabilizer와 연결되는가?

이 관계를 이용해 임의의 정상 논리 상태가 복원되는지 설명하세요. 실제 오류 이름과 보정 이름이 다르다는 사실만으로 실패라고 판단할 수 있는지도 함께 적습니다.

### 3a 답안

계산 결과와 근거:

…

### (b) 신드롬만 사용하는 decoder 작성하기

`decode_syndrome(syndrome)`를 작성하세요.

입력 `syndrome`은 무오류 또는 단일 데이터 Pauli 오류에서 나타날 수 있는 <strong>길이 8의 정수 tuple</strong>이라고 보장합니다. 순서는 앞 문제와 같이

```python
(a0, a1, a2, a3, b0, b1, b2, b3)
```

입니다.

`single_error_candidates()`가 제공하는 후보들을 조사하고, 앞에서 작성한 `predict_syndrome()`를 사용해 받은 신드롬과 일치하는 후보 하나를 찾으세요.

반환값은 다음과 같은 `(Pauli, 데이터 번호)` tuple입니다.

```python
("X", 4)
```

아무 연산도 하지 않는 보정은 다음과 같이 반환합니다.

```python
("I", None)
```

두 함수에서 사용하는 입력 형식을 구분하세요.

- `single_error_candidates()`의 원소는 <strong>후보 하나</strong>를 나타내는 tuple입니다.
- `predict_syndrome()`는 <strong>오류들의 목록 또는 tuple</strong>을 받습니다.

따라서 일반적인 단일 후보를 조사할 때에는 `[candidate]`처럼 후보 하나를 목록에 담아 전달합니다. 무오류 후보 `("I", None)`를 조사할 때에는 `predict_syndrome(())`처럼 빈 오류 목록을 사용하세요.

같은 신드롬의 후보가 여러 개이면 처음이나 마지막 후보 등 어느 것을 선택해도 됩니다. 반환하는 보정은 제공된 28개 후보 중 하나여야 합니다. 입력 보장 밖의 신드롬에 대해서는 예외를 내도 됩니다.

<strong>실제 오류나 처음 준비한 논리 상태를 함수에 추가로 전달하지 마세요.</strong> 보정 선택에는 받은 신드롬과 코드의 후보 정보만 사용합니다.

In [ ]:
def decode_syndrome(syndrome):
    # TODO BEGIN — 신드롬으로 무오류 또는 단일 오류 보정 후보를 선택하세요.
    raise NotImplementedError("decode_syndrome")
    # TODO END

In [ ]:
_ = check_decode_syndrome(decode_syndrome)

### 3b 답안

계산 결과와 근거:

…

### (c) 무오류와 모든 단일 오류에서 정정 확인하기

작성한 decoder를 제공된 반복 실험에 연결하세요. 무오류와 단일 X·Y·Z 오류 27개 각각에 대해 다음 항목을 확인합니다.

- 실제 측정으로 얻은 신드롬
- decoder가 선택한 보정
- 보정 후 신드롬
- 제공 확인 셀의 논리 정보 보존 검사 결과

전체 결과를 확인한 뒤, <strong>실제 오류와 다른 보정이 선택된 사례</strong>를 하나 골라 다음 표를 완성하세요.

| 실제 오류 $E$ | 측정 신드롬 | 선택한 보정 $R$ | 남는 연산 $RE$ | 논리 상태가 복원되는 이유 |
|---|---|---|---|---|
| … | … | … | … | … |

그 차이가 어떤 stabilizer 작용인지 설명하세요. 같은 신드롬에서 다른 동등 후보를 선택하는 decoder도 정답이 될 수 있는 이유를 적습니다.

마지막으로, 이번 실험에서 확인한 정정 보장의 조건을 정리하세요. 실제 오류가 어떤 범위에 있었는지, 검사와 보정에 어떤 가정을 사용했는지 포함해야 합니다. <strong>신드롬이 모두 0이라는 결과만을 논리 정보 복원의 근거로 삼지 말고</strong>, 제공된 상태 보존 검사와 앞의 연산 관계를 함께 사용하세요.

다음 문제에서는 실제 오류가 이 가정을 벗어났을 때 같은 decoder가 어떤 보정을 선택하는지 살펴보겠습니다. 보정 후 검사 결과와 원래 논리 상태에 대한 일치도를 따로 계산해, 성공이라는 판단이 무엇에 근거해야 하는지 확인하겠습니다.

### 실행 3c — 작성한 함수로 결과 확인

실행 전 예상을 적고 일반 실습 코드 실행 설정을 True로 바꾸어 다시 실행하세요. 앞의 코드 검사도 통과하면 이 실험이 실행됩니다. 결과를 확인한 뒤 답안에 실행 결과와 설명을 남기세요.

In [ ]:
def experiment_3c():
    rows, results = [], []
    for error in single_error_candidates():
        errors = [] if error[0] == "I" else [error]
        trial = recovery_trial("+i", errors, decode_syndrome)
        rows.append([error, trial["syndrome_before"], trial["correction"], trial["syndrome_after"]])
        results.append({"error": error, **trial})
    show_table(["실제 오류", "측정 신드롬", "학생이 고른 보정", "보정 후 신드롬"], rows)
    print("상태 보존은 위의 보정 선택 확인 셀에서 두 논리 기저에 대한 공통 위상까지 검사합니다.")
    return results

_ = run_demo("3c", experiment_3c, ("2b · 신드롬 예측", "3b · 보정 선택"))

### 3c 답안

계산 결과와 근거:

…

## 문제 4. 신드롬이 정상으로 돌아오면 정정에 성공한 걸까요?

앞 문제에서는 최대 한 데이터 Pauli 오류라는 가정 아래 보정 규칙을 만들었습니다. 이번에는 <strong>보정 후의 상태가 처음 저장한 논리 정보와 같은지 직접 계산</strong>합니다.

먼저 모든 검사를 통과하는 상태와 처음 저장한 상태가 왜 다를 수 있는지 살펴보겠습니다. 이어서 데이터 두 개에 오류를 넣고, 앞에서 작성한 decoder를 그대로 적용한 결과를 여러 논리 입력에서 비교하겠습니다.

### 정상 코드 공간과 처음의 논리 상태는 다릅니다

정상 코드 공간에는 $|0_L\rangle$뿐 아니라 $|1_L\rangle$와 그 중첩도 들어 있습니다. 이 상태들은 모두 여덟 stabilizer의 검사값이 $+1$입니다.

따라서 신드롬이 모두 0이라는 사실은 <strong>데이터가 정상 검사 조건을 만족하는 코드 공간에 있다</strong>는 것을 알려 줍니다. 그러나 그 공간 안에서 처음과 같은 논리 상태인지는 아직 알 수 없습니다.

예를 들어 처음에 $|0_L\rangle$를 저장했는데 출력이 $|1_L\rangle$로 바뀌었다고 합시다. 두 상태 모두 정상 검사 조건을 만족하므로 신드롬만 확인하면 차이가 드러나지 않습니다. 하지만 저장한 정보는 달라졌습니다.

이처럼 코드 공간 안에서 논리 상태를 바꾸는 연산을 <strong>논리 연산</strong>이라고 부릅니다. 논리 X는 두 논리 기저를 서로 바꾸는 연산입니다.

$$
\overline X|0_L\rangle=|1_L\rangle,
\qquad
\overline X|1_L\rangle=|0_L\rangle.
$$

이번 패치에서는

$$
\overline X=X_0X_3X_6
$$

을 사용할 수 있습니다. 이 연산은 모든 stabilizer와 교환하므로 정상 신드롬을 유지하면서도 논리 기저를 서로 바꿉니다.

앞 문제에서 보았던 stabilizer의 작용과 구분하세요. Stabilizer는 정상 논리 상태를 모두 그대로 두지만, 논리 X는 정상 코드 공간을 유지하면서 그 안의 정보를 바꿉니다.

논리 상태 사이의 상대위상을 바꾸는 연산도 있습니다. 논리 Z는

$$
\overline Z|0_L\rangle=|0_L\rangle,
\qquad
\overline Z|1_L\rangle=-|1_L\rangle
$$

로 정의합니다. 따라서 논리 정보를 보존하려면 논리 기저의 값뿐 아니라, 두 기저 사이의 상대위상도 유지해야 합니다.

### 어떤 입력에서는 남은 오류가 드러나지 않을 수 있습니다

논리 X가 남으면 $|0_L\rangle$는 $|1_L\rangle$로 바뀝니다. 그러나 모든 입력이 같은 방식으로 달라지는 것은 아닙니다.

논리 $|+_L\rangle$를 생각해 보겠습니다.

$$
|+_L\rangle
=\frac{|0_L\rangle+|1_L\rangle}{\sqrt2}.
$$

이 상태에 논리 X를 적용하면

$$
\begin{aligned}
\overline X|+_L\rangle
&=\frac{\overline X|0_L\rangle+\overline X|1_L\rangle}{\sqrt2}\\
&=\frac{|1_L\rangle+|0_L\rangle}{\sqrt2}\\
&=|+_L\rangle
\end{aligned}
$$

입니다. 두 성분의 순서만 바뀌므로 같은 상태로 남습니다.

따라서 $|+_L\rangle$ 하나만 시험하면 남은 논리 X의 영향을 놓칠 수 있습니다. 이 결과는 <strong>그 입력이 보존되었다는 뜻</strong>이지, 보정 과정이 모든 논리 입력을 보존한다는 뜻은 아닙니다.

이 점은 앞 문제에서 두 논리 기저에 대한 작용을 같은 전역위상으로 비교했던 이유와 연결됩니다. 임의의 논리 정보를 보호하려면 특정 상태 하나에서만 변화가 보이지 않는 것으로는 충분하지 않습니다.

### 목표 상태와 출력 상태의 fidelity 계산하기

이제 처음 상태와 보정 후 상태가 얼마나 일치하는지 수치로 확인하겠습니다.

이번 실험에서는 정규화된 순수 논리 입력에 <strong>정해진 Pauli 오류</strong>를 적용합니다. 각 실험에서 신드롬은 하나로 정해지며, 이상적인 측정과 보정을 마친 출력도 순수상태 벡터로 나타낼 수 있습니다.

목표 논리 상태를 $|\psi_L\rangle$, 보정 후 상태를 $|\phi\rangle$라고 하면, 두 상태의 fidelity는

$$
F=|\langle\psi_L|\phi\rangle|^2
$$

입니다.

상태벡터의 성분을 각각 $\psi_j,\phi_j$라고 쓰면 내적은

$$
\langle\psi_L|\phi\rangle
=\sum_j\psi_j^*\phi_j
$$

입니다. 별표는 켤레복소수를 뜻합니다. <strong>목표 벡터의 성분에 켤레복소수를 취하고, 출력 벡터의 대응하는 성분과 곱해 더한 뒤, 그 절댓값을 제곱</strong>하면 됩니다.

정규화된 두 상태에서 fidelity는 0과 1 사이입니다.

| Fidelity | 의미 |
|---|---|
| $F=1$ | 두 상태가 전역위상을 제외하고 같습니다. |
| $F=0$ | 두 상태가 서로 직교합니다. |
| $0<F<1$ | 두 상태가 일부 겹치지만 같지는 않습니다. |

상대위상을 가진 상태도 검사해야 하므로 켤레복소수를 빠뜨리지 않아야 합니다. 또한 내적의 절댓값과 그 제곱은 다른 값이라는 점에 주의하세요.

### 함께 확인하기: 전역위상만 다른 출력

보정 후 상태가 처음 상태에 $i$만 곱해진

$$
|\phi\rangle=i|\psi_L\rangle
$$

라고 합시다. 상태가 정규화되어 있으므로

$$
\langle\psi_L|\phi\rangle
=i\langle\psi_L|\psi_L\rangle=i
$$

이고,

$$
F=|i|^2=1
$$

입니다.

두 벡터의 성분을 그대로 비교하면 다르게 보이지만, 물리적인 상태는 같습니다. 따라서 이런 출력을 정정 실패로 판단해서는 안 됩니다.

여기서 사용하는 식은 <strong>두 입력이 순수상태 벡터인 경우</strong>의 fidelity입니다. 측정 결과를 구분하지 않고 섞어 얻은 일반적인 혼합상태에 같은 벡터 식을 그대로 적용하지는 않습니다.

### 검증 도구가 아는 정보와 decoder가 아는 정보 구분하기

실습에서는 정정 성능을 시험하기 위해 어떤 논리 상태를 준비했는지 알고 있습니다. 제공 도구는 그 목표 상태와 보정 후 상태를 반환하고, 여러분이 작성한 함수가 두 상태의 fidelity를 계산합니다.

그러나 목표 상태를 알고 있는 것은 <strong>실험을 준비하고 검증하는 도구</strong>입니다. 이 정보가 decoder에 전달되는 것은 아닙니다. Decoder는 계속 신드롬만 받아 보정을 선택합니다.

제공 도구

```python
recovery_trial(logical_label, errors, decode_syndrome)
```

는 다음 과정을 수행합니다.

1. `logical_label`에 해당하는 논리 상태를 준비합니다.
2. `errors`에 지정한 오류를 적용합니다.
3. 이상적인 stabilizer 측정으로 신드롬을 얻습니다.
4. 그 신드롬만 `decode_syndrome()`에 전달합니다.
5. 반환된 보정을 적용하고 결과를 정리합니다.

이번 문제에서 사용할 반환 항목은 다음과 같습니다.

| 항목 | 의미 |
|---|---|
| `target` | 처음 준비한 논리 상태 벡터 |
| `corrected` | 선택된 보정을 적용한 뒤의 상태 벡터 |
| `syndrome_before` | 보정 전 측정 신드롬 |
| `correction` | decoder가 실제로 선택한 보정 |
| `syndrome_after` | 보정 후의 신드롬 |

전체 데이터 상태는 $|q_8\cdots q_0\rangle$ 순서의 512개 성분을 사용합니다. 이 벡터를 직접 전개하거나 인코더를 작성할 필요는 없습니다. 제공된 두 벡터에 여러분의 fidelity 함수를 적용하면 됩니다.

### 오류가 두 개이면 같은 decoder는 무엇을 할까요?

이제 q0와 q3에 X 오류를 적용하겠습니다.

$$
E=X_0X_3.
$$

코드 상태 준비, 검사, 보정 연산은 여전히 이상적입니다. 검사표와 decoder도 그대로 유지합니다. 바뀌는 것은 <strong>실제 데이터 오류가 한 개에서 두 개로 늘어났다는 점</strong>입니다.

Decoder는 실제 오류 수를 전달받지 않습니다. 받은 신드롬과 일치하는 단일 오류 후보가 있으면, 앞에서 작성한 규칙에 따라 그 후보를 선택합니다.

이때 “단일 오류 후보와 신드롬이 일치했다”는 사실은 “실제로 단일 오류가 발생했다”는 증거가 아닙니다. 서로 다른 오류가 같은 신드롬을 만들 수 있기 때문입니다.

따라서 이번에는 다음을 각각 확인해야 합니다.

- 어떤 신드롬이 나왔는가?
- Decoder가 어떤 보정을 골랐는가?
- 보정 후 검사 결과가 정상으로 돌아왔는가?
- 보정 후 상태가 처음 논리 상태와 같은가?

앞의 세 항목과 마지막 항목을 구분해서 기록하겠습니다.

### 직접 풀어보기

### (a) 논리 상태의 복원 검사 작성하기

`logical_fidelity(target_state, corrected_state)`를 작성하세요.

두 인자는 같은 길이의 정규화된 복소 순수상태 벡터입니다. 반환값은 두 상태의 fidelity를 나타내는 실수 하나입니다.

전역위상만 다른 상태, 서로 직교하는 상태, 상대위상이 있는 상태에 모두 같은 계산이 적용되어야 합니다. 특정 입력이나 512차원에서만 맞는 값을 반환하지 마세요.

함수를 완성한 뒤 다음 두 출력을 비교하세요. 목표 상태는 $|0_L\rangle$입니다.

| 출력 상태 | Stabilizer 검사 결과 | 목표 상태에 대한 fidelity |
|---|---|---|
| $\lvert0_L\rangle$ | … | … |
| $\lvert1_L\rangle$ | … | … |

두 검사가 각각 무엇을 확인하는지 설명하세요. 모든 검사값이 정상인 상태들 사이에서도 fidelity가 달라질 수 있는 이유를 코드 공간과 논리 정보의 관계로 정리합니다.

In [ ]:
def logical_fidelity(target_state, corrected_state):
    # TODO BEGIN — 두 정규화된 순수 상태의 fidelity를 반환하세요.
    raise NotImplementedError("logical_fidelity")
    # TODO END

In [ ]:
_ = check_logical_fidelity(logical_fidelity)

### 실행 4a — 작성한 함수로 결과 확인

실행 전 예상을 적고 일반 실습 코드 실행 설정을 True로 바꾸어 다시 실행하세요. 앞의 코드 검사도 통과하면 이 실험이 실행됩니다. 결과를 확인한 뒤 답안에 실행 결과와 설명을 남기세요.

In [ ]:
def experiment_4a():
    target = logical_state("0")
    results, rows = [], []
    for label in ("0", "1"):
        output = logical_state(label)
        syndrome = deterministic_syndrome(output)
        score = logical_fidelity(target, output)
        rows.append([label, syndrome, score])
        results.append({"output_label": label, "syndrome": syndrome, "fidelity": score})
    show_table(["출력 논리 상태", "신드롬", "목표 논리 0에 대한 F"], rows)
    return results

_ = run_demo("4a", experiment_4a, ("4a · 논리 fidelity",))

### 4a 답안

계산 결과와 근거:

…

### (b) 두 오류에 기존 decoder 적용하기

오류 목록을 다음과 같이 정합니다.

```python
errors = [("X", 0), ("X", 3)]
```

논리 입력 `"0"`과 `"+"`를 각각 실험하세요. 문제 3의 `decode_syndrome()`는 수정하지 않습니다.

실행 도구가 반환한 `target`과 `corrected`에 여러분의 `logical_fidelity()`를 적용해 다음 표를 완성합니다.

| 논리 입력 | 보정 전 신드롬 | 선택된 보정 | 보정 후 신드롬 | 원래 입력에 대한 $F$ |
|---|---|---|---|---|
| $\lvert0_L\rangle$ | … | … | … | … |
| $\lvert+_L\rangle$ | … | … | … | … |

실행 전에 어떤 결과를 예상했는지 적고, 실행 후에는 예상과 비교하세요.

이어서 실제 오류 $E=X_0X_3$ 뒤에 선택된 보정 $R$을 적용했을 때 남는 연산 $RE$를 적으세요. 그 연산을 논리 연산 또는 stabilizer와 연결해, 두 논리 입력의 결과를 설명합니다.

같은 신드롬에서 동등한 다른 단일 오류 후보를 선택했다면 결과가 어떻게 되는지도 생각해 보세요. 보정 이름이 달라지는 것과 논리 상태에 남는 작용이 달라지는 것을 구분해야 합니다.

### 4b 답안

실행 전 예상:

…

계산·실행 결과와 근거:

…

### 실행 4b — 작성한 함수로 결과 확인

실행 전 예상을 적고 일반 실습 코드 실행 설정을 True로 바꾸어 다시 실행하세요. 앞의 코드 검사도 통과하면 이 실험이 실행됩니다. 결과를 확인한 뒤 답안에 실행 결과와 설명을 남기세요.

In [ ]:
def experiment_4b():
    errors = [("X",0), ("X",3)]
    results, rows = [], []
    for label in ("0", "+"):
        trial = recovery_trial(label, errors, decode_syndrome)
        score = logical_fidelity(trial["target"], trial["corrected"])
        results.append({"logical_label": label, "fidelity": score, **trial})
        rows.append([label, trial["syndrome_before"], trial["correction"], trial["syndrome_after"], score])
    show_table(["논리 입력", "보정 전 신드롬", "선택된 보정", "보정 후 신드롬", "F"], rows)
    return results

_ = run_demo("4b", experiment_4b, ("2b · 신드롬 예측", "3b · 보정 선택", "4a · 논리 fidelity"))

### (c) 정정 성공 보고서 검토하기

다음 보고서를 (b)의 결과에 비추어 검토하세요.

> 보정 후 신드롬이 모두 0으로 돌아왔습니다. 논리 + 입력에서는 fidelity도 1이므로, 이 보정 규칙이 원래의 논리 정보를 복원했다고 결론 내렸습니다.

이 보고서가 확인한 사실과, 그 사실만으로는 결론 낼 수 없는 내용을 나누어 설명하세요.

- 신드롬이 모두 0이라는 결과는 무엇을 확인한 것인가요?
- 논리 + 입력에서 fidelity가 1이라는 결과는 무엇을 확인한 것인가요?
- 논리 0의 결과까지 함께 보면, 임의의 논리 정보가 복원되었다고 말할 수 있나요?
- 두 입력의 차이는 보정 뒤 남은 연산과 어떻게 연결되나요?

이어서 이번 결과가 문제 3의 단일 오류 정정 보장과 모순되는지 설명하세요. 앞 문제의 가정과 이번 실험에서 바뀐 조건을 정확히 적습니다.

<strong>임의의 두 오류를 정정한다고 보장하지 않는다는 것과, 모든 두 오류가 반드시 정정에 실패한다는 것은 다른 주장</strong>입니다. 이번에 실제로 시험한 오류에서 어떤 결과를 얻었는지와, 코드 전체에 대해 무엇이 보장되는지를 구분해서 보고하세요.

### 4c 답안

계산 결과와 근거:

…

## 마지막 적용 — 두 오류를 Z형으로 바꾸면 무엇이 달라질까요?

같은 코드, 같은 이상적인 측정, 같은 decoder를 유지하고 오류 목록만 다음과 같이 바꿉니다.

```python
errors = [("Z", 0), ("Z", 1)]
```

이번에도 실제 오류는 데이터 두 개에 적용됩니다. 새로운 decoder를 만들거나 기존 보정 규칙을 수정하지 마세요.

실행 전에 신드롬과 단일 오류 보정 후보를 예상하세요. 이어서 논리 입력 `"0"`과 `"+"`를 각각 실험하고, 앞에서 작성한 fidelity 함수로 결과를 확인합니다.

| 논리 입력 | 예상한 보정 후보 | 실제 선택된 보정 | 보정 후 신드롬 | 원래 입력에 대한 $F$ |
|---|---|---|---|---|
| $\lvert0_L\rangle$ | … | … | … | … |
| $\lvert+_L\rangle$ | … | … | … | … |

앞의 X형 두 오류 사례와 비교해 다음 내용을 설명하세요.

- 보정 뒤 검사 결과가 정상으로 돌아오나요?
- 어느 논리 입력에서 남은 오류가 드러나고, 어느 입력에서는 가려지나요?
- 실제 오류와 선택된 보정을 합친 연산은 논리 상태에 어떤 작용을 하나요?

남는 연산을 구한 뒤에는 <strong>그 연산이 두 논리 기저에 각각 어떻게 작용하는지</strong> 확인해 보세요. 제공 함수 `logical_state("0")`, `logical_state("1")`로 기저 상태를 얻고, `apply_errors(상태, 오류_목록)`으로 실제 오류와 선택한 보정을 순서대로 적용할 수 있습니다. 각 입력과 출력의 내적을 비교하면, 상태 전체에 붙는 위상도 확인할 수 있습니다.

두 기저가 각각 원래 방향으로 돌아오더라도, 두 출력에 <strong>같은 위상</strong>이 붙었는지, <strong>서로 다른 상대위상</strong>이 남았는지를 구분해야 합니다. 같은 위상이라면 임의의 중첩도 그 공통 전역위상만 남기고 복원됩니다. 상대위상이 다르면 중첩은 바뀔 수 있습니다. 출력이 다른 기저로 옮겨진 경우도 함께 구분하세요. 앞의 두 fidelity를 이 기저별 작용과 연결해 설명하면 됩니다.

새 함수나 회로를 작성할 필요는 없습니다. 앞에서 만든 도구를 재사용해, <strong>어떤 입력 하나가 보존되었다는 사실과 임의의 논리 정보를 복원한다는 주장의 차이</strong>를 짧은 설명으로 정리하세요.

## 마지막 적용 답안

실행 전 예상:

…

계산·실행 결과와 근거:

…

### 적용 결과 확인

예상과 근거를 먼저 적은 뒤 아래 셀의 `RUN_APPLICATION`을 `True`로 바꾸어 실행하세요. 앞에서 작성한 함수로 결과를 계산합니다. 이 실행 설정은 새로운 함수 TODO가 아닙니다.

In [ ]:
RUN_APPLICATION = False


def application_experiment():
    errors = [("Z",0), ("Z",1)]
    results, rows = [], []
    for label in ("0", "+"):
        prediction = predict_syndrome(errors)
        trial = recovery_trial(label, errors, decode_syndrome)
        score = logical_fidelity(trial["target"], trial["corrected"])
        rows.append([label, prediction, trial["syndrome_before"], trial["correction"], trial["syndrome_after"], score])
        results.append(dict(trial, logical_label=label, fidelity=score, prediction=prediction))
    show_table(["논리 입력", "학생의 예측", "측정 신드롬", "선택 보정", "보정 후 신드롬", "F"], rows)
    return results


LAB_RESULTS.pop("application", None)
if RUN_APPLICATION:
    _ = run_demo("application", application_experiment,
                 ("2b · 신드롬 예측", "3b · 보정 선택", "4a · 논리 fidelity"), requires_observations=False)
else:
    print("실행 전 예상을 먼저 적으세요. 확인하려면 RUN_APPLICATION을 True로 바꾸고 이 셀을 실행하세요.")

## Lab 3 마무리

네 코드 검사의 상태와 각 소문항의 설명 칸을 확인하세요. 마지막으로 오류 정정의 성공을 판단할 때 확인할 조건을 세 문장으로 정리하세요. 검사 결과와 사후 상태, decoder가 사용하는 정보와 오류 가정, 신드롬 복구와 논리 상태 복원의 차이를 생각해 보세요.

### 마무리 답안

계산 결과와 근거:

…

In [ ]:
expected_checks = ['1b · stabilizer 측정', '2b · 신드롬 예측', '3b · 보정 선택', '4a · 논리 fidelity']
show_table(["코드 확인", "상태"], [[name, CHECK_STATUS.get(name, "미실행")] for name in expected_checks])
if all(CHECK_STATUS.get(name) == "통과" for name in expected_checks):
    print("네 코드 검사를 모두 통과했습니다. 서술형 설명과 마지막 적용 답안은 별도로 확인하세요.")
else:
    print("미완성·미실행·실패 항목을 확인하세요. 수정한 답안 셀과 확인 셀, 뒤의 연결 실험을 다시 실행하세요.")